# baseline_colab.ipynb — measure unmodified scikit-image v0.22.0

> **IMPORTANT — Colab runtime.** Before running:
>   1. `Runtime → Change runtime type → Hardware accelerator: CPU`
>   2. **`Runtime version: 2025.07`** (Python 3.11). v0.20.0 does NOT
>      have a Python 3.12 wheel — Latest runtime will fail to install.
>   3. Click Save. If prompted, disconnect+delete the old runtime
>      and reconnect to the new one.
>
> Then `Runtime → Run all`. Whole notebook is ~10–15 minutes.


## What this notebook does
1. Runtime info.
2. Clone this submission repo.
3. Install scikit-image v0.22.0 from PyPI (binary wheel).
4. Build the workload.
5. Run `regionprops_table` once, save as golden output.
6. Run the existing scikit-image regionprops tests, save pass-set.
7. Benchmark baseline timing.


## 1. Runtime info

In [ ]:
!cat /proc/cpuinfo | grep "model name" | head -1
!free -h
!python --version


## 2. Clone the submission repo

In [ ]:
# ====================================================================
# CONFIG  (usually nothing to change)
# --------------------------------------------------------------------
# SUBMISSION_REPO_URL = the public repo holding the patch, workload
# generator, and benchmark utils. Override via env var if forking.
# ====================================================================
import os
SUBMISSION_REPO_URL = os.environ.get(
    "SUBMISSION_REPO_URL",
    "https://github.com/Sanjay040695/scikit-image-performance-optimization.git"
)
SUBMISSION_DIR = "/content/submission"
print("Using submission repo:", SUBMISSION_REPO_URL)

# Always re-clone so we pick up any push you made after a previous run.
!rm -rf $SUBMISSION_DIR
!git clone --depth 1 $SUBMISSION_REPO_URL $SUBMISSION_DIR

import sys
for p in (SUBMISSION_DIR,
          os.path.join(SUBMISSION_DIR, "workloads"),
          os.path.join(SUBMISSION_DIR, "benchmark"),
          os.path.join(SUBMISSION_DIR, "patches")):
    if p not in sys.path:
        sys.path.insert(0, p)
print("OK — submission repo at:", SUBMISSION_DIR)


## 3. Install scikit-image v0.22.0

In [ ]:
# ====================================================================
# Install scikit-image v0.24.0 (June 2024 — ~23 months old, comfortably
# >12 months per the brief). Its wheel is numpy-2.x ABI compatible, so
# we do NOT touch numpy/scipy — those are kept at Colab's defaults,
# which avoids the in-memory-vs-on-disk mismatches that earlier
# numpy upgrades created.
# ====================================================================
import sys, subprocess, os

# Install (or downgrade) scikit-image to exactly v0.24.0. Nothing else.
# Hard-stop if on Python 3.12 — v0.20.0 has no Python 3.12 wheel.
if sys.version_info[:2] >= (3, 12):
    raise SystemExit(
        "ERROR: this notebook needs Colab runtime '2025.07' (Python 3.11). "
        "Go to Runtime > Change runtime type > Runtime version > '2025.07', "
        "then Runtime > Disconnect and delete runtime, then Run all."
    )

# v0.20.0 was built against numpy 1.x ABI. If Colab's runtime has
# numpy 2.x loaded, the import will fail. Detect this up front and
# restart the kernel cleanly if needed.
_need_restart = False
if "numpy" in sys.modules:
    import numpy as _np
    if int(_np.__version__.split(".")[0]) >= 2:
        _need_restart = True
        print(f"Detected numpy {_np.__version__} (2.x) already loaded — "
              "will restart kernel after install for a clean 1.x environment.")

# Pin numpy<2.0 (compatible with v0.20.0 wheel)
!pip install -q --upgrade "numpy<2.0" "scipy<1.13" 2>&1 | tail -3
!pip install -q --upgrade --no-deps "scikit-image==0.20.0" 2>&1 | tail -3

if _need_restart:
    print()
    print("=" * 60)
    print("Restarting kernel for clean numpy 1.x load.")
    print("Colab will auto-reconnect — when it does, click")
    print("`Runtime > Run all` ONCE more and the rest of the notebook")
    print("will run cleanly.")
    print("=" * 60)
    os.kill(os.getpid(), 9)

# Force-reload so any previously imported skimage is dropped.
for k in list(sys.modules):
    if k.startswith("skimage"):
        del sys.modules[k]

import numpy, scipy, skimage
print(f"numpy:   {numpy.__version__}")
print(f"scipy:   {scipy.__version__}")
print(f"skimage: {skimage.__version__}    (expected 0.20.0)")
assert skimage.__version__ == "0.20.0", (
    f"Expected scikit-image 0.20.0 but got {skimage.__version__}. "
    "Install did not take effect."
)

# Record the SHA so reward.json can reference it.
BASELINE_SHA = subprocess.run(
    ["git", "ls-remote", "https://github.com/scikit-image/scikit-image.git", "refs/tags/v0.20.0"],
    capture_output=True, text=True,
).stdout.split()[0]
print(f"baseline tag v0.20.0 resolves to SHA: {BASELINE_SHA}")
with open("/content/baseline_sha.txt", "w") as f:
    f.write(BASELINE_SHA)

print()
print("OK — environment is ready.")


## 4. Build the workload

In [ ]:
from generate_workload import build_workload, WORKLOAD_PROPERTIES
import time, numpy as np, os
t0 = time.perf_counter()
labels, intensities, n_regions = build_workload()
print(f"workload: shape={labels.shape}, n_regions={n_regions}, "
      f"channels={len(intensities)} "
      f"({time.perf_counter()-t0:.2f}s)")
os.makedirs("/content/outputs", exist_ok=True)
np.savez_compressed("/content/outputs/workload.npz",
                    labels=labels, intensity=intensities[0])


## 5. Golden output (channel 0)

In [ ]:
from skimage.measure import regionprops_table
out = regionprops_table(labels, intensity_image=intensities[0],
                        properties=list(WORKLOAD_PROPERTIES))
np.savez_compressed("/content/outputs/golden_output.npz", **out)
print("golden_output.npz columns:", sorted(out.keys()))
print("first 3 areas:", out["area"][:3])


## 6. Existing test suite — record pass-set

In [ ]:
import subprocess, json, skimage
test_dir = os.path.join(os.path.dirname(skimage.__file__),
                        "measure", "tests")
proc = subprocess.run(
    ["pytest", os.path.join(test_dir, "test_regionprops.py"),
     "-v", "--tb=no", "-q", "--no-header"],
    capture_output=True, text=True,
)
print(proc.stdout[-2000:])
passed, failed = [], []
for line in proc.stdout.splitlines():
    if " PASSED" in line:
        passed.append(line.split(" PASSED")[0].strip())
    elif " FAILED" in line:
        failed.append(line.split(" FAILED")[0].strip())
with open("/content/outputs/baseline_tests.json", "w") as f:
    json.dump({"passed": sorted(passed), "failed": sorted(failed)},
              f, indent=2)
print(f"\n{len(passed)} pass, {len(failed)} fail.")


## 7. Benchmark baseline

In [ ]:
from benchmark_utils import bench, write_json, collect_environment
# Workload = iterate regionprops_table over ALL intensity channels.
# This is the realistic multi-channel pipeline workload and the
# total wall-clock time naturally exceeds the brief's 10s threshold.
def run_baseline():
    for intensity in intensities:
        regionprops_table(labels, intensity_image=intensity,
                          properties=list(WORKLOAD_PROPERTIES))
baseline_stats = bench(run_baseline, label="baseline",
                       n_warmup=2, n_measured=7)
write_json("/content/outputs/benchmark_results.json", {
    "baseline_time_s": baseline_stats,
    "environment": collect_environment(),
    "baseline_sha": BASELINE_SHA,
})
print(f"\nBaseline median: {baseline_stats['median']:.3f}s "
      f"(IQR {baseline_stats['iqr']:.3f}s)")
